# Tasks

1. ข้อมูลมา 2 sources คือ SETSMART, Eikon แต่ละ source มีข้อมูลอะไรเท่าไหร่ ขาดอะไร มีอะไรระบุต่างกัน เอามารวมกันได้ไหม ทุก column เลย
2. SETSMART ดึงได้แค่ 5 ปี (2021-2025) ข้อมูล meeting จะหายไป 15 ปี ทำยังไงดี
3. save ลง csv พร้อมเข้า stata

# Phase A — Eikon Cleaning (port from `clean workflow.do`)

อ่านไฟล์ Excel จาก Eikon 14 ไฟล์ + Age 1 ไฟล์ → reshape เป็น long panel → merge เป็น master panel เดียว

**Eikon Excel layout** ทุกไฟล์ (ยกเว้น Age):
- Row 0: ชื่อ column ซ้ำ (Company Common Name | Country | Variable | Variable | ...)
- Row 1: `46130` (Excel serial date ของวันที่ pull data ≈ May 2026)
- Row 2: `FY0, FY-1, FY-2, ...` (lag year labels — บางไฟล์มี NaN gap คั่น)
- Row 3+: ข้อมูลจริง (col A=ticker, B=name, C=country, D+=ค่าต่อ FY)

In [21]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW = Path(r"d:/SAVER/Research/CG_QandPBV_TH/raw and clean data - Copy")
OUT = Path(r"d:/SAVER/Research/CG_QandPBV_TH/raw and clean data")

FY0_CALENDAR_YEAR = 2025
MAX_LAG = 20

## A1. Generic Eikon panel cleaner

ฟังก์ชันกลางใช้กับไฟล์ panel ทุกไฟล์ — รับ path, sheet, ชื่อตัวแปร แล้วคืน long DataFrame `(ticker, company_name, country, lag_years, <var>)`

วิธีจัดการ FY gap: ใช้ row 2 (FY label row) ในการ map column → lag_years จริง ไม่ใช่ rename ตามลำดับเฉยๆ (ปลอดภัยกว่า Stata workflow เพราะ Stata renames sequentially แล้วจะ map ผิดถ้ามี gap)

In [22]:
def clean_eikon_panel(file_path: Path, sheet: str, var_name: str,
                     force_numeric: bool = True, max_lag: int | None = MAX_LAG) -> pd.DataFrame:
    """Read Eikon-format Excel and return long panel: (ticker, company_name, country, lag_years, <var_name>).

    Auto-detects 2 vs 3 header-row layout:
      - 3-header (most files): row 0=label, row 1=date, row 2=FY labels, row 3+=data
      - 2-header (Mkt Cap):    row 0=label, row 1=date,                  row 2+=data
    In 2-header layout, sort row-1 dates descending to assign lag 0, 1, 2, ...
    Drops duplicate-lag columns (some files repeat the FY block twice).
    """
    raw = pd.read_excel(file_path, sheet_name=sheet, header=None)

    # Detect layout: is row 2 col 3 an FY label?
    probe = raw.iloc[2, 3] if raw.shape[1] > 3 else None
    has_fy_row = isinstance(probe, str) and bool(re.match(r"FY(-?\d*)", probe.strip()))

    col_to_lag: dict[int, int] = {}
    if has_fy_row:
        # 3-header: map via row 2 FY labels (skip NaN gap cols, skip duplicate FY blocks)
        for col_offset in range(3, raw.shape[1]):
            fy_label = raw.iloc[2, col_offset]
            if pd.isna(fy_label):
                continue
            m = re.match(r"FY(-?\d*)", str(fy_label).strip())
            if not m:
                continue
            token = m.group(1)
            lag = abs(int(token)) if token else 0
            if lag in col_to_lag.values():  # skip repeated FY block
                continue
            col_to_lag[col_offset] = lag
        data_start = 3
    else:
        # 2-header: row 1 has per-column dates; sort desc -> assign lag 0, 1, 2, ...
        dated = []
        for col_offset in range(3, raw.shape[1]):
            d = raw.iloc[1, col_offset]
            if pd.notna(d):
                dated.append((col_offset, float(d)))
        dated.sort(key=lambda x: -x[1])  # most recent first
        for lag, (col_offset, _) in enumerate(dated):
            col_to_lag[col_offset] = lag
        data_start = 2

    # Slice data rows, drop rows with missing ticker
    df = raw.iloc[data_start:].copy().reset_index(drop=True)
    df = df.rename(columns={0: "ticker", 1: "company_name", 2: "country"})
    df = df.dropna(subset=["ticker"])

    # Rename FY columns -> <var>_<lag>
    rename_map = {col: f"{var_name}_{lag}" for col, lag in col_to_lag.items()}
    df = df.rename(columns=rename_map)
    df = df[["ticker", "company_name", "country"] + list(rename_map.values())]

    # Stata: duplicates drop ticker, force (keep first)
    df = df.drop_duplicates(subset=["ticker"], keep="first")

    # Reshape long
    long_df = df.melt(
        id_vars=["ticker", "company_name", "country"],
        value_vars=list(rename_map.values()),
        var_name="_lag_str",
        value_name=var_name,
    )
    long_df["lag_years"] = long_df["_lag_str"].str.extract(rf"{re.escape(var_name)}_(\d+)").astype(int)
    long_df = long_df.drop(columns="_lag_str")

    if force_numeric:
        long_df[var_name] = pd.to_numeric(long_df[var_name], errors="coerce")

    if max_lag is not None:
        long_df = long_df[long_df["lag_years"] < max_lag].reset_index(drop=True)

    return long_df.sort_values(["ticker", "lag_years"]).reset_index(drop=True)

## A2. File config

(file, sheet, var_name, numeric?) — ตามลำดับใน `clean workflow.do`

In [23]:
EIKON_PANEL_FILES = [
    # (file_name, sheet_name, var_name, force_numeric)
    ("total asset-copy .xlsx",                  "Sheet2",                  "assets",                  True),
    ("ESG Combined score grade-copy.xlsx",      "Sheet1",                  "esg_grade",               False),
    ("ESG Combined Score-copy.xlsx",            None,                      "esg_score",               True),
    ("Board Size-copy.xlsx",                    "Board Size-clean",        "boardsize",               True),
    ("Independent Board Member %-copy.xlsx",    None,                      "ind_member_pct",          True),
    ("Num of Board Meeting per Year-copy.xlsx", None,                      "num_meeting_py",          True),
    ("Board Meeting AVG Attendance %-copy.xlsx",None,                      "avg_attent_meeting_pct",  True),
    ("Leverage%-copy.xlsx",                     None,                      "leverage",                True),
    ("PBV-copy.xlsx",                           "PBV-clean",               "pbv",                     True),
    ("Mkt Cap - For Q-copy.xlsx",               None,                      "mktcapQ",                 True),
    ("Total Lib - For Q-copy.xlsx",             None,                      "libforQ",                 True),
    ("Assets Reported - For Q-copy.xlsx",       None,                      "assetsforQ",              True),
    ("CEO Duality-copy.xlsx",                   "CEO Duality-clean",       "dua",                     True),
]

def resolve_sheet(file_path: Path, sheet: str | None) -> str:
    """Use given sheet or fall back to first sheet in workbook."""
    if sheet is not None:
        return sheet
    return pd.ExcelFile(file_path).sheet_names[0]

## A3. Run cleaner on all 13 panel files

In [24]:
panels: dict[str, pd.DataFrame] = {}
for fname, sheet, var, numeric in EIKON_PANEL_FILES:
    fp = RAW / fname
    sheet_resolved = resolve_sheet(fp, sheet)
    df = clean_eikon_panel(fp, sheet_resolved, var, force_numeric=numeric)
    panels[var] = df
    n_tickers = df['ticker'].nunique()
    n_obs = len(df)
    print(f"  {var:30s} -> shape {df.shape}, tickers {n_tickers}, lag_years {sorted(df['lag_years'].unique())[:3]}..{sorted(df['lag_years'].unique())[-3:]}")

  assets                         -> shape (17560, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(17), np.int64(18), np.int64(19)]
  esg_grade                      -> shape (15804, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(15), np.int64(16), np.int64(17)]
  esg_score                      -> shape (15804, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(15), np.int64(16), np.int64(17)]
  boardsize                      -> shape (15804, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(15), np.int64(16), np.int64(17)]
  ind_member_pct                 -> shape (15804, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(15), np.int64(16), np.int64(17)]
  num_meeting_py                 -> shape (15804, 5), tickers 878, lag_years [np.int64(0), np.int64(1), np.int64(2)]..[np.int64(15), np.int64(16), np.int64(17)]
  avg_attent_meeting_pct         -

## A4. Clean Age (cross-sectional, ไม่ reshape)

Age file: row 0 = header (Company Common Name | Country | Date of Incorporation), row 1+ = data ตรงๆ

In [25]:
age_raw = pd.read_excel(RAW / "Age-copy.xlsx", sheet_name="Age-clean", header=None)
age = age_raw.iloc[1:].copy().reset_index(drop=True)
age.columns = ["ticker", "company_name", "country", "incorp_date"]
age = age.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"], keep="first")
age["incorp_date"] = pd.to_datetime(age["incorp_date"], errors="coerce")
age["incorp_year"] = age["incorp_date"].dt.year
print(f"Age: shape {age.shape}, tickers {age['ticker'].nunique()}")
age.head()

Age: shape (878, 5), tickers 878


,ticker,company_name,country,incorp_date,incorp_year
0,AJ.BK,AJ Plast PCL,Thailand,1987-03-25,1987.0
1,ADVANC.BK,Advanced Info Service PCL,Thailand,1992-11-13,1992.0
2,AHC.BK,Aikchol Hospital PCL,Thailand,1978-11-15,1978.0
3,ALUCON.BK,Alucon PCL,Thailand,1961-04-25,1961.0
4,AMARIN.BK,Amarin Corporations PCL,Thailand,1993-06-15,1993.0


## A5. Merge → Eikon master panel

Sequential left-merge on `(ticker, lag_years)` ไปยังทุก panel แล้ว m:1 กับ Age บน `ticker` เท่านั้น

เพิ่ม:
- `calendar_year = FY0_CALENDAR_YEAR - lag_years` (FY0=2025 → year)
- `incorp_age = calendar_year - incorp_year`

In [26]:
# Start with the first panel (assets) as base
master_eikon = panels["assets"].copy()

for var, df in panels.items():
    if var == "assets":
        continue
    # Only merge the value column to avoid duplicate company_name/country
    master_eikon = master_eikon.merge(
        df[["ticker", "lag_years", var]],
        on=["ticker", "lag_years"],
        how="outer",
    )

# Merge Age (m:1 on ticker)
master_eikon = master_eikon.merge(
    age[["ticker", "incorp_date", "incorp_year"]],
    on="ticker",
    how="left",
)

# Calendar year and age
master_eikon["calendar_year"] = FY0_CALENDAR_YEAR - master_eikon["lag_years"]
master_eikon["incorp_age"] = master_eikon["calendar_year"] - master_eikon["incorp_year"]

print("master_eikon shape:", master_eikon.shape)
print("unique tickers:   ", master_eikon["ticker"].nunique())
print("lag_years range:  ", master_eikon["lag_years"].min(), "-", master_eikon["lag_years"].max())
print("calendar_year:    ", master_eikon["calendar_year"].min(), "-", master_eikon["calendar_year"].max())
print("columns:\n", master_eikon.columns.tolist())

master_eikon shape: (17560, 21)
unique tickers:    878
lag_years range:   0 - 19
calendar_year:     2006 - 2025
columns:
 ['ticker', 'company_name', 'country', 'assets', 'lag_years', 'esg_grade', 'esg_score', 'boardsize', 'ind_member_pct', 'num_meeting_py', 'avg_attent_meeting_pct', 'leverage', 'pbv', 'mktcapQ', 'libforQ', 'assetsforQ', 'dua', 'incorp_date', 'incorp_year', 'calendar_year', 'incorp_age']


## A6. Quick sanity check

In [27]:
# Non-null counts per variable
value_cols = [c for c in master_eikon.columns if c not in ["ticker", "company_name", "country", "lag_years", "calendar_year", "incorp_date", "incorp_year", "incorp_age"]]
print("Non-null counts per variable:")
print(master_eikon[value_cols].notna().sum().sort_values(ascending=False))
print()
print("Sample rows for ADVANC.BK:")
master_eikon[master_eikon["ticker"] == "ADVANC.BK"].sort_values("lag_years").head(10)

Non-null counts per variable:
leverage                  13958
libforQ                   13942
assets                    13942
mktcapQ                   12444
pbv                       12103
assetsforQ                 2669
ind_member_pct             1348
esg_score                  1348
esg_grade                  1348
dua                        1348
boardsize                  1346
num_meeting_py             1331
avg_attent_meeting_pct     1321
dtype: int64

Sample rows for ADVANC.BK:


,ticker,company_name,country,assets,lag_years,esg_grade,esg_score,boardsize,ind_member_pct,num_meeting_py,...,leverage,pbv,mktcapQ,libforQ,assetsforQ,dua,incorp_date,incorp_year,calendar_year,incorp_age
260,ADVANC.BK,Advanced Info Service PCL,Thailand,4.202732e+11,0,B+,68.054244,12.0,41.666667,10.0,...,74.47366,8.685397,9.309276e+11,3.129928e+11,NaN,0.0,1992-11-13,1992.0,2025,33.0
261,ADVANC.BK,Advanced Info Service PCL,Thailand,4.314321e+11,1,B+,69.884113,11.0,33.333333,8.0,...,77.44167,8.779230,8.535982e+11,3.341082e+11,4.314321e+11,0.0,1992-11-13,1992.0,2024,32.0
262,ADVANC.BK,Advanced Info Service PCL,Thailand,4.544392e+11,2,A-,75.913655,11.0,41.666667,12.0,...,80.04613,7.124956,6.454035e+11,3.637610e+11,4.544392e+11,0.0,1992-11-13,1992.0,2023,31.0
263,ADVANC.BK,Advanced Info Service PCL,Thailand,3.370437e+11,3,B+,71.482350,11.0,33.333333,13.0,...,74.53851,6.767868,5.799709e+11,2.512273e+11,3.370437e+11,0.0,1992-11-13,1992.0,2022,30.0
264,ADVANC.BK,Advanced Info Service PCL,Thailand,3.562217e+11,4,B+,71.505046,11.0,45.454545,11.0,...,77.02989,8.372466,6.840029e+11,2.743972e+11,4.202732e+11,0.0,1992-11-13,1992.0,2021,29.0
265,ADVANC.BK,Advanced Info Service PCL,Thailand,3.501706e+11,5,B+,73.296071,11.0,45.454545,9.0,...,78.38500,6.926898,5.233456e+11,2.744812e+11,3.562217e+11,0.0,1992-11-13,1992.0,2020,28.0
266,ADVANC.BK,Advanced Info Service PCL,Thailand,2.896691e+11,6,B+,74.592156,10.0,45.454545,9.0,...,76.04364,9.142265,6.332870e+11,2.202750e+11,3.501706e+11,0.0,1992-11-13,1992.0,2019,27.0
267,ADVANC.BK,Advanced Info Service PCL,Thailand,2.905050e+11,7,B,65.268807,11.0,46.666667,8.0,...,80.14885,8.915738,5.128589e+11,2.328364e+11,2.896691e+11,0.0,1992-11-13,1992.0,2018,26.0
268,ADVANC.BK,Advanced Info Service PCL,Thailand,2.840674e+11,8,B,62.750141,13.0,38.461538,9.0,...,82.24830,11.284559,5.678612e+11,2.336406e+11,2.905050e+11,0.0,1992-11-13,1992.0,2017,25.0
269,ADVANC.BK,Advanced Info Service PCL,Thailand,2.756704e+11,9,B+,68.111891,13.0,38.888889,11.0,...,84.50746,10.266443,4.370450e+11,2.329620e+11,2.840674e+11,0.0,1992-11-13,1992.0,2016,24.0


## A7. ยืนยันว่า column ตัวเลขใหญ่เป็น `numeric` ไม่ใช่ `string`

ค่าที่ขึ้น `4.2e+11` เป็น Python float (scientific notation แค่การแสดงผล) ไม่ใช่ string — sanity check ผ่าน 3 วิธี: dtype, type ของ sample, math op

In [28]:
# 1) Dtypes ของทุก column
print("=== Dtypes ของทุก column ===")
print(master_eikon.dtypes.to_string())
print()

# 2) เน้น 4 column ตัวเลขใหญ่: assets, mktcapQ, libforQ, assetsforQ
print("=== ยืนยันว่า 4 column เงินบาทเป็น numeric ===")
for col in ["assets", "mktcapQ", "libforQ", "assetsforQ"]:
    s = master_eikon[col]
    sample = s.dropna().iloc[0]
    print(f"  {col:12s}  dtype={str(s.dtype):10s}  is_numeric={pd.api.types.is_numeric_dtype(s)}  sample={sample}  type(sample)={type(sample).__name__}")
print()

# 3) Math operations - ถ้าทำได้แสดงว่าเป็น numeric จริง
print("=== Math test (ถ้าทำได้ = numeric) ===")
print(f"  assets.mean()  = {master_eikon['assets'].mean():,.2f}  baht")
print(f"  mktcapQ.max()  = {master_eikon['mktcapQ'].max():,.2f}  baht")
print(f"  libforQ.sum()  = {master_eikon['libforQ'].sum():,.2f}  baht")

=== Dtypes ของทุก column ===
ticker                               str
company_name                         str
country                              str
assets                           float64
lag_years                          int64
esg_grade                            str
esg_score                        float64
boardsize                        float64
ind_member_pct                   float64
num_meeting_py                   float64
avg_attent_meeting_pct           float64
leverage                         float64
pbv                              float64
mktcapQ                          float64
libforQ                          float64
assetsforQ                       float64
dua                              float64
incorp_date               datetime64[us]
incorp_year                      float64
calendar_year                      int64
incorp_age                       float64

=== ยืนยันว่า 4 column เงินบาทเป็น numeric ===
  assets        dtype=float64     is_numeric=True  sample=2379

## A8. การกระจายตัวของข้อมูล numeric

In [29]:
master_eikon.describe().round(2)

,assets,lag_years,esg_score,boardsize,ind_member_pct,num_meeting_py,avg_attent_meeting_pct,leverage,pbv,mktcapQ,libforQ,assetsforQ,dua,incorp_date,incorp_year,calendar_year,incorp_age
count,1.394200e+04,17560.00,1348.00,1346.00,1348.00,1331.00,1321.00,13958.00,12103.00,1.244400e+04,1.394200e+04,2.669000e+03,1348.00,17480,17480.00,17560.00,17480.00
mean,4.189905e+10,9.50,52.94,12.01,43.60,9.65,96.12,37.75,2.25,1.996094e+10,2.804575e+10,1.618168e+11,0.13,1997-07-24 05:01:30.617849,1997.07,2015.50,18.43
min,1.000000e+04,0.00,0.42,5.00,0.00,1.00,74.15,-65070.59,0.00,9.749902e+05,-1.926790e+13,2.859543e+08,0.00,1913-12-08 00:00:00,1913.00,2006.00,-17.00
25%,1.142056e+09,4.75,41.34,10.00,33.33,7.00,94.53,24.88,0.73,9.100000e+08,3.235862e+08,5.846299e+09,0.00,1988-04-05 00:00:00,1988.00,2010.75,5.00
50%,2.992730e+09,9.50,54.75,12.00,41.42,9.00,97.40,44.33,1.18,2.334223e+09,1.142072e+09,2.027983e+10,0.00,1995-12-24 00:00:00,1995.00,2015.50,18.00
75%,9.705045e+09,14.25,65.58,14.00,50.00,12.00,99.31,61.61,2.09,7.921804e+09,4.342259e+09,7.072100e+10,0.00,2012-01-01 00:00:00,2012.00,2020.25,29.00
max,4.606342e+12,19.00,91.73,28.00,94.74,29.00,100.00,15616.04,1871.55,2.157970e+12,4.030659e+12,4.606342e+12,1.00,2023-03-24 00:00:00,2023.00,2025.00,112.00
std,2.535271e+11,5.77,17.15,3.13,11.65,4.07,4.43,679.06,21.25,7.946364e+10,2.790240e+11,5.313642e+11,0.34,NaN,16.49,5.77,17.47


# Phase B — Ticker Harmonization

Parse Eikon ticker → 6 columns ใหม่:
- `symbol` (bare, ตัด suffix ทุกอย่างออก)
- `market` (`SET` / `mai`)
- `delisted` (True/False)
- `delisted_code` (เช่น `F24`, `L24`)
- `is_trust` (True ถ้า ticker ลงท้าย `u.BK` — REIT/Trust/Fund unit)
- `namechange` (True ถ้า ticker มี `_n` หรือ `_mn` — Eikon marker หลัง rename)

**Eikon convention** (จาก 878 tickers):
| pattern | count | example |
|---|---|---|
| `XX.BK` | 602 | `AJ.BK` → SET |
| `XXm.BK` | 188 | `DV8m.BK` → mai |
| `XXu.BK` | 54 | `AIMCGu.BK` → SET trust/REIT |
| `XX_n.BK` | 2 | `APCO_n.BK`, `ITEL_n.BK` → SET namechange |
| `XX_mn.BK` | 3 | `SGF_mn.BK`, `SK_mn.BK`, `TIGER_mn.BK` → mai namechange |
| `XX.BK^XYZ` | 29 | `BKI.BK^F24` → delisted (can be SET, mai, or trust) |

## B1. ฟังก์ชัน parse Eikon ticker

Regex: `^(.+?)(m|_mn)?(_n|u)?\.BK(\^[A-Z0-9]+)?$`
- group 1 = symbol (non-greedy match)
- group 2 = `m` (mai) หรือ `_mn` (mai+namechange)
- group 3 = `_n` (namechange) หรือ `u` (trust/REIT)
- group 4 = `^XXX` (delisted)

In [30]:
TICKER_PATTERN = re.compile(r"^(.+?)(m|_mn)?(_n|u)?\.BK(\^[A-Z0-9]+)?$")

def parse_eikon_ticker(ticker: str) -> dict:
    """Parse Eikon ticker -> {symbol, market, delisted, delisted_code, is_trust, namechange}."""
    m = TICKER_PATTERN.match(str(ticker))
    if not m:
        return {"symbol": ticker, "market": "unknown", "delisted": False, "delisted_code": None,
                "is_trust": False, "namechange": False}
    mkt_suffix = m.group(2) or ""
    spec_suffix = m.group(3) or ""
    delisted_match = m.group(4)
    return {
        "symbol": m.group(1),
        "market": "mai" if "m" in mkt_suffix else "SET",   # 'm' or '_mn' both contain 'm'
        "delisted": delisted_match is not None,
        "delisted_code": delisted_match[1:] if delisted_match else None,
        "is_trust": spec_suffix == "u",
        "namechange": (mkt_suffix == "_mn") or (spec_suffix == "_n"),
    }

# Quick test (covers all suffix patterns)
for t in ["AJ.BK", "DV8m.BK", "BKI.BK^F24", "ALLm.BK^I24", "3K-BAT.BK^L24",
          "AIMCGu.BK", "M-PATu.BK", "APCO_n.BK", "ITEL_n.BK",
          "TIGER_mn.BK", "SGF_mn.BK"]:
    print(f"  {t:18s} -> {parse_eikon_ticker(t)}")

  AJ.BK              -> {'symbol': 'AJ', 'market': 'SET', 'delisted': False, 'delisted_code': None, 'is_trust': False, 'namechange': False}
  DV8m.BK            -> {'symbol': 'DV8', 'market': 'mai', 'delisted': False, 'delisted_code': None, 'is_trust': False, 'namechange': False}
  BKI.BK^F24         -> {'symbol': 'BKI', 'market': 'SET', 'delisted': True, 'delisted_code': 'F24', 'is_trust': False, 'namechange': False}
  ALLm.BK^I24        -> {'symbol': 'ALL', 'market': 'mai', 'delisted': True, 'delisted_code': 'I24', 'is_trust': False, 'namechange': False}
  3K-BAT.BK^L24      -> {'symbol': '3K-BAT', 'market': 'SET', 'delisted': True, 'delisted_code': 'L24', 'is_trust': False, 'namechange': False}
  AIMCGu.BK          -> {'symbol': 'AIMCG', 'market': 'SET', 'delisted': False, 'delisted_code': None, 'is_trust': True, 'namechange': False}
  M-PATu.BK          -> {'symbol': 'M-PAT', 'market': 'SET', 'delisted': False, 'delisted_code': None, 'is_trust': True, 'namechange': False}
  APCO_n.

## B2. เพิ่ม 4 column ใหม่เข้า `master_eikon`

In [31]:
parsed = master_eikon["ticker"].apply(parse_eikon_ticker).apply(pd.Series)
master_eikon = pd.concat([master_eikon, parsed], axis=1)

# Reorder: ticker info up front
front_cols = ["ticker", "symbol", "market", "delisted", "delisted_code", "is_trust", "namechange",
              "company_name", "country", "lag_years", "calendar_year"]
other_cols = [c for c in master_eikon.columns if c not in front_cols]
master_eikon = master_eikon[front_cols + other_cols]

print("master_eikon shape:", master_eikon.shape)
print()
print("Firm-level ticker breakdown:")
firm_lookup = master_eikon[["ticker", "symbol", "market", "delisted", "is_trust", "namechange"]].drop_duplicates("ticker")
print(firm_lookup.groupby(["market", "delisted", "is_trust", "namechange"]).size().to_string())
print()
print(f"Total unique tickers : {firm_lookup['ticker'].nunique()}")
print(f"Total unique symbols : {firm_lookup['symbol'].nunique()}")

master_eikon shape: (17560, 27)

Firm-level ticker breakdown:
market  delisted  is_trust  namechange
SET     False     False     False         602
                            True            2
                  True      False          54
        True      False     False          13
                  True      False          14
mai     False     False     False         188
                            True            3
        True      False     False           2

Total unique tickers : 878
Total unique symbols : 878


## B3. สร้าง `eikon_symbol_lookup` (canonical reference สำหรับ Phase C/D)

Table: `symbol → (market, delisted, delisted_code, eikon_ticker, latest_company_name)` — ใช้ map SETSMART symbols ให้รู้ว่าเป็น SET/mai/delisted

In [32]:
eikon_symbol_lookup = (
    master_eikon[["symbol", "market", "delisted", "delisted_code", "is_trust", "namechange",
                  "ticker", "company_name"]]
    .drop_duplicates("ticker")
    .rename(columns={"ticker": "eikon_ticker", "company_name": "latest_company_name"})
    .sort_values(["symbol", "delisted"])
    .reset_index(drop=True)
)
print("eikon_symbol_lookup shape:", eikon_symbol_lookup.shape)
print()
# Check duplicate symbols (same symbol on both markets, or active+delisted)
dup_symbols = eikon_symbol_lookup[eikon_symbol_lookup.duplicated("symbol", keep=False)].sort_values("symbol")
print(f"Symbols with multiple rows: {dup_symbols['symbol'].nunique()}")
print(dup_symbols.head(20).to_string(index=False))

eikon_symbol_lookup shape: (878, 8)

Symbols with multiple rows: 0
Empty DataFrame
Columns: [symbol, market, delisted, delisted_code, is_trust, namechange, eikon_ticker, latest_company_name]
Index: []


## B4. ตัด Trust/REIT/Fund ออกจาก `master_eikon`

Drop `is_trust=True` ออกจาก `master_eikon` (panel data ที่จะใช้ analyze)

**หมายเหตุ**: เก็บ `eikon_symbol_lookup` ไว้ครบ (รวม trust) เพื่อให้ C3 จับ SETSMART symbols ที่เป็น trust ได้ถูกต้อง — ส่วน C5 ค่อย drop trust ออกจาก `master_setsmart` ที่หลัง

In [33]:
# Report trust counts before dropping
n_trust_tickers = eikon_symbol_lookup['is_trust'].sum()
n_trust_rows = master_eikon['is_trust'].sum()
print(f"Eikon has {n_trust_tickers} trust tickers ({n_trust_rows} firm-year rows)")

# Drop trust ONLY from master_eikon (the panel for Phase D analysis)
master_eikon = master_eikon[~master_eikon['is_trust']].reset_index(drop=True)

# Intentionally keep eikon_symbol_lookup with trusts — so C3 can flag SETSMART trust symbols
# (e.g., SETSMART "DREIT" -> Eikon "DREITu.BK" is_trust=True, then C5 drops it from master_setsmart)
print()
print(f"After drop: master_eikon {master_eikon.shape}")
print(f"  unique tickers in master_eikon: {master_eikon['ticker'].nunique()}")
print(f"  unique symbols in master_eikon: {master_eikon['symbol'].nunique()}")
print()
print(f"eikon_symbol_lookup retained intact ({len(eikon_symbol_lookup)} rows incl. trusts for SETSMART cross-ref)")

Eikon has 68 trust tickers (1360 firm-year rows)

After drop: master_eikon (16200, 27)
  unique tickers in master_eikon: 810
  unique symbols in master_eikon: 810

eikon_symbol_lookup retained intact (878 rows incl. trusts for SETSMART cross-ref)


In [34]:
master_eikon

,ticker,symbol,market,delisted,delisted_code,is_trust,namechange,company_name,country,lag_years,...,avg_attent_meeting_pct,leverage,pbv,mktcapQ,libforQ,assetsforQ,dua,incorp_date,incorp_year,incorp_age
0,2S.BK,2S,SET,False,NaN,False,False,2S Metal PCL,Thailand,0,...,NaN,14.40801,0.630932,1.279255e+09,342856000.0,NaN,NaN,1992-05-29,1992.0,33.0
1,2S.BK,2S,SET,False,NaN,False,False,2S Metal PCL,Thailand,1,...,NaN,7.39988,0.784848,1.517989e+09,154321000.0,NaN,NaN,1992-05-29,1992.0,32.0
2,2S.BK,2S,SET,False,NaN,False,False,2S Metal PCL,Thailand,2,...,NaN,7.71362,0.829877,1.671988e+09,168179000.0,NaN,NaN,1992-05-29,1992.0,31.0
3,2S.BK,2S,SET,False,NaN,False,False,2S Metal PCL,Thailand,3,...,NaN,11.83655,0.828606,1.671988e+09,270586000.0,NaN,NaN,1992-05-29,1992.0,30.0
4,2S.BK,2S,SET,False,NaN,False,False,2S Metal PCL,Thailand,4,...,NaN,7.31134,1.195560,2.724989e+09,179676000.0,NaN,NaN,1992-05-29,1992.0,29.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16195,ZIGAm.BK,ZIGA,mai,False,NaN,False,False,Ziga Innovation PCL,Thailand,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1998-04-21,1998.0,12.0
16196,ZIGAm.BK,ZIGA,mai,False,NaN,False,False,Ziga Innovation PCL,Thailand,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1998-04-21,1998.0,11.0
16197,ZIGAm.BK,ZIGA,mai,False,NaN,False,False,Ziga Innovation PCL,Thailand,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1998-04-21,1998.0,10.0
16198,ZIGAm.BK,ZIGA,mai,False,NaN,False,False,Ziga Innovation PCL,Thailand,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1998-04-21,1998.0,9.0


# Phase C — SETSMART Cleaning

3 dataset จาก SETSMART:
1. **Meeting attendance** (สรุปประชุม BoD ต่อปี) — รวม `summary_board_attendance.csv` (807 บริษัท main, สร้างไว้แล้ว) + re-aggregate `output/*.csv` (52 บริษัทใหม่ที่ summary เดิมยังไม่ได้รวม)
2. **ESG Comparison** Governance section — stack 5 ไฟล์ (2564-2568 = 2021-2025), cols 110-177 = 68 sub-indicators
3. **Listing reference** จาก `symbols_company_year.xlsx` — บอกว่าปีไหน symbol นั้น listed อยู่

## C1. Meeting Attendance

Port logic จาก `aggregate_attendance.py`:
- กรองเฉพาะ Section = "Meeting Attendance of the Board of Directors"
- ข้าม "No Information", "0 / 0"
- รวม attended/should_attend ของทุกกรรมการในปีนั้น
- คำนวณ `% = total_attended / total_should_attend × 100`
- ใช้ `Board Meeting Count` ใน CSV เป็น `Total_Board_Meetings_Per_Year` ก่อน, ถ้าไม่มี fallback ไปใช้ max(should_attend) ของกรรมการที่ active ทั้งปี

**Strategy**: ใช้ `summary_board_attendance.csv` (aggregated แล้ว 807 บริษัท) + re-aggregate 52 บริษัทใหม่ใน `output/`

In [35]:
from urllib.parse import unquote

def parse_meeting_fraction(text):
    """'24 / 24' -> (24, 24), 'N/A' -> None, '' -> None."""
    if not text or str(text).strip() in ("", "N/A"):
        return None
    cleaned = str(text).replace(" ", "")
    parts = cleaned.split("/")
    if len(parts) != 2:
        return None
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return None


def aggregate_one_meeting_file(filepath: Path) -> list[dict]:
    """Port of aggregate_attendance.process_one_file() — pandas-friendly.

    URL-decodes the Symbol (e.g., F%26D -> F&D) since SETSMART scraper writes
    URL-encoded symbols when the company name contains special characters.
    """
    df = pd.read_csv(filepath, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    if df.empty:
        return []

    raw_symbol = df["Symbol"].iloc[0] if "Symbol" in df.columns and df["Symbol"].iloc[0] else                  filepath.name.replace("_meeting_attendance.csv", "")
    symbol = unquote(raw_symbol)  # F%26D -> F&D

    # Only Board of Directors section
    bod_all = df[df["Section"].str.contains("Board of Directors", na=False)].copy()
    if bod_all.empty:
        return []
    bod = bod_all[~bod_all["Name"].isin(["No Information", "No Data", ""])]

    # Years to report (include "No Info" years as N/A)
    all_years = set()
    for y in bod_all["Year"]:
        try: all_years.add(int(y))
        except ValueError: pass

    results = []
    for year in sorted(all_years):
        year_rows = bod[bod["Year"] == str(year)]

        if year_rows.empty:
            results.append({"Company_Symbol": symbol, "Year": year,
                           "Total_Board_Meetings_Per_Year": None,
                           "Company_Overall_Attendance": None,
                           "Total_Should_Attend": None,
                           "Company_Overall_Attendance_Percentage": None})
            continue

        # Total Board Meetings from CSV's "Board Meeting Count" column
        board_meetings = 0
        bmc = year_rows["Board Meeting Count"].iloc[0].strip()
        if bmc and bmc != "N/A":
            try: board_meetings = int(bmc)
            except ValueError: pass

        total_attended = 0
        total_should = 0
        valid_members = 0
        for _, m in year_rows.iterrows():
            frac = parse_meeting_fraction(m.get("Number of Board Meeting", ""))
            if frac is None: continue
            att, should = frac
            if att == 0 and should == 0: continue
            total_attended += att
            total_should += should
            valid_members += 1
            # Fallback for board_meetings if not in CSV column
            if board_meetings == 0:
                term = m.get("Termination Date", "").strip()
                if term in ("N/A", "", "-"):
                    board_meetings = max(board_meetings, should)

        if valid_members == 0:
            results.append({"Company_Symbol": symbol, "Year": year,
                           "Total_Board_Meetings_Per_Year": None,
                           "Company_Overall_Attendance": None,
                           "Total_Should_Attend": None,
                           "Company_Overall_Attendance_Percentage": None})
            continue

        pct = round(total_attended / total_should * 100, 2) if total_should > 0 else 0.0
        results.append({"Company_Symbol": symbol, "Year": year,
                       "Total_Board_Meetings_Per_Year": board_meetings,
                       "Company_Overall_Attendance": total_attended,
                       "Total_Should_Attend": total_should,
                       "Company_Overall_Attendance_Percentage": pct})
    return results


# (a) Load existing aggregated summary (807 main tickers)
summary_existing = pd.read_csv(RAW / "summary_board_attendance.csv", encoding="utf-8-sig")
print(f"Existing summary: {summary_existing.shape}, unique tickers: {summary_existing['Company_Symbol'].nunique()}")

# (b) Aggregate new tickers from output/ (includes URL-encoded names like F%26D)
new_csvs = sorted((RAW / "output").glob("*_meeting_attendance.csv"))
print(f"New CSVs to aggregate: {len(new_csvs)}")
new_records = []
for fp in new_csvs:
    new_records.extend(aggregate_one_meeting_file(fp))
summary_new = pd.DataFrame(new_records)
print(f"New aggregated: {summary_new.shape}, unique tickers: {summary_new['Company_Symbol'].nunique() if not summary_new.empty else 0}")

# (c) Concat
setsmart_meeting = pd.concat([summary_existing, summary_new], ignore_index=True)
setsmart_meeting = setsmart_meeting.rename(columns={
    "Company_Symbol": "symbol",
    "Year": "year",
    "Total_Board_Meetings_Per_Year": "num_meeting_setsmart",
    "Company_Overall_Attendance": "total_attended_setsmart",
    "Total_Should_Attend": "total_should_attend_setsmart",
    "Company_Overall_Attendance_Percentage": "avg_attent_meeting_pct_setsmart",
})
# Cast numerics
for col in ["num_meeting_setsmart", "total_attended_setsmart", "total_should_attend_setsmart", "avg_attent_meeting_pct_setsmart"]:
    setsmart_meeting[col] = pd.to_numeric(setsmart_meeting[col], errors="coerce")
setsmart_meeting["year"] = pd.to_numeric(setsmart_meeting["year"], errors="coerce").astype("Int64")

# Filter to 2021-2025 only (FY0=2025 overlap window)
setsmart_meeting = setsmart_meeting[setsmart_meeting["year"].between(2021, 2025)].reset_index(drop=True)

print()
print(f"setsmart_meeting (combined, 2021-2025): {setsmart_meeting.shape}")
print(f"unique symbols: {setsmart_meeting['symbol'].nunique()}")
print()
setsmart_meeting.head(10)


Existing summary: (4842, 6), unique tickers: 807
New CSVs to aggregate: 55
New aggregated: (330, 6), unique tickers: 55

setsmart_meeting (combined, 2021-2025): (4310, 6)
unique symbols: 861



,symbol,year,num_meeting_setsmart,total_attended_setsmart,total_should_attend_setsmart,avg_attent_meeting_pct_setsmart
0,2S,2021,NaN,NaN,NaN,NaN
1,2S,2022,5.0,50.0,50.0,100.00
2,2S,2023,4.0,40.0,40.0,100.00
3,2S,2024,6.0,59.0,60.0,98.33
4,2S,2025,5.0,49.0,50.0,98.00
5,A5,2021,NaN,NaN,NaN,NaN
6,A5,2022,7.0,42.0,42.0,100.00
7,A5,2023,5.0,30.0,30.0,100.00
8,A5,2024,11.0,66.0,66.0,100.00
9,A5,2025,5.0,33.0,35.0,94.29


## C2. ESG Comparison — Board Composition (Empirical column mapping)

**Background**: SETSMART export มี bug — row 11 header labels ไม่ตรงกับ data ในแต่ละ column (ตามที่เปิด Excel แล้ว user confirm)

**Solution**: reverse-engineer column meanings โดยใช้ ratio test (`male / total * 100 == pct_male`) ซึ่ง verify ใน 12 บริษัท × 4 ปี (2022-2025) ผ่าน 100%

**Files**: ใช้ `- New` files ที่ user re-pulled (5 ไฟล์: 2564-2568 = 2021-2025)

**Coverage**: 2022-2025 ครบ; 2021 (2564) มีข้อมูลแค่ ~6 บริษัท (SETSMART เพิ่งเริ่มเก็บ Composition data ตั้งแต่ปี 2022)

**Columns ที่เก็บ** (cols 122-129 ของ Excel, verified empirically):

| Excel col | meaning | type |
|---|---|---|
| 122 | total_directors | count |
| 123 | male_directors | count |
| 124 | pct_male | % |
| 125 | female_directors | count |
| 126 | pct_female | % |
| 127 | **independent_directors** | count — CG var |
| 128 | **pct_independent** | % — CG var |
| 129 | male_independent | count |

In [36]:
ESG_FILES_NEW = {
    2021: ["ESG Comparison 2564 - New.xlsx",
           "ESG Comparison F&D, L&E, S&F 2564.xlsx"],  # special file with F&D, L&E, S&J (filename typo S&F)
    2022: ["ESG Comparison 2565 - New.xlsx",
           "ESG Comparison F&D, L&E, S&F 2565.xlsx"],
    2023: ["ESG Comparison 2566 - New.xlsx",
           "ESG Comparison F&D, L&E, S&F 2566.xlsx"],
    2024: ["ESG Comparison 2567 - New.xlsx",
           "ESG Comparison F&D, L&E, S&F 2567.xlsx"],
    2025: ["ESG Comparison 2568 - New.xlsx",
           "ESG Comparison F&D, L&E, S&F 2568.xlsx"],
}

# Empirical column mapping (verified via ratio test across 12 firms x 4 years)
BOARD_COMP_COLS = {
    122: "total_directors_setsmart",
    123: "male_directors_setsmart",
    124: "pct_male_setsmart",
    125: "female_directors_setsmart",
    126: "pct_female_setsmart",
    127: "independent_directors_setsmart",
    128: "pct_independent_setsmart",
    129: "male_independent_setsmart",
}

def find_data_start(raw: pd.DataFrame) -> int:
    """Find row index where ESG data starts (just after 'Symbol' header row + 1 spacer).
    Regular files: Symbol at row 10 -> data starts row 12.
    Special files (only 3 firms): Symbol at row 9 -> data starts row 11.
    """
    for r in range(min(20, raw.shape[0])):
        if str(raw.iloc[r, 0]).strip() == "Symbol":
            return r + 2
    raise ValueError("Cannot find 'Symbol' header in ESG file")


def read_esg_board_comp(file_path: Path, expected_year: int) -> pd.DataFrame:
    """Read ESG Comparison file, extract symbol + year + Board composition cols (empirical mapping)."""
    raw = pd.read_excel(file_path, sheet_name="Sheet1", header=None)
    data_start = find_data_start(raw)
    df = raw.iloc[data_start:, [0, 1] + list(BOARD_COMP_COLS.keys())].copy()
    df.columns = ["symbol", "year"] + list(BOARD_COMP_COLS.values())
    df = df.dropna(subset=["symbol"])
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df[df["year"] == expected_year].reset_index(drop=True)
    for col in BOARD_COMP_COLS.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


# Stack all year files (regular + special per year)
stacked = []
for year, fnames in ESG_FILES_NEW.items():
    year_dfs = []
    for fname in fnames:
        df = read_esg_board_comp(RAW / fname, year)
        year_dfs.append(df)
    yr_combined = pd.concat(year_dfs, ignore_index=True).drop_duplicates(subset=["symbol", "year"], keep="first")
    n_with_data = yr_combined["total_directors_setsmart"].notna().sum()
    print(f"  {year}: shape {yr_combined.shape}, firms with total_directors: {n_with_data}")
    stacked.append(yr_combined)

setsmart_board = pd.concat(stacked, ignore_index=True)
print()
print(f"setsmart_board (combined 2021-2025): {setsmart_board.shape}")
print(f"unique symbols: {setsmart_board['symbol'].nunique()}")
print()
print("Coverage by year (firms with non-null total_directors_setsmart):")
print(setsmart_board.groupby("year")["total_directors_setsmart"].apply(lambda x: x.notna().sum()).to_string())
print()
print("New special-file firms:")
print(setsmart_board[setsmart_board["symbol"].isin(["F&D", "L&E", "S&J"])].sort_values(["symbol","year"]).to_string())


  2021: shape (855, 10), firms with total_directors: 6
  2022: shape (888, 10), firms with total_directors: 811
  2023: shape (896, 10), firms with total_directors: 844
  2024: shape (880, 10), firms with total_directors: 861
  2025: shape (775, 10), firms with total_directors: 775

setsmart_board (combined 2021-2025): (4294, 10)
unique symbols: 907

Coverage by year (firms with non-null total_directors_setsmart):
year
2021      6
2022    811
2023    844
2024    861
2025    775

New special-file firms:
     symbol  year  total_directors_setsmart  male_directors_setsmart  pct_male_setsmart  female_directors_setsmart  pct_female_setsmart  independent_directors_setsmart  pct_independent_setsmart  male_independent_setsmart
232     F&D  2021                       NaN                      NaN                NaN                        NaN                  NaN                             NaN                       NaN                        NaN
1094    F&D  2022                      11.0       

## C3. Enrich SETSMART symbols ด้วย market/delisted จาก Eikon (+ manual rename mapping)

SETSMART ไม่บอก SET/mai/delisted → ใช้ `eikon_symbol_lookup` (จาก Phase B) เป็น reference

**Manual rename mapping**: บางบริษัทเปลี่ยนชื่อ ticker — SETSMART ใช้ชื่อปัจจุบัน, Eikon เก็บชื่อเก่า (ส่วนใหญ่เป็น delisted):

| SETSMART | Eikon (alias) | reason |
|---|---|---|
| `SCB` | `SCBB` | SCBB delisted Dec 2022 → SCBX restructure → SCB |
| `TRUE` | `TRUEE` | TRUEE delisted Mar 2023 → merger DTAC → TRUE |
| `STECON` | `STEC` | STEC delisted Oct 2024 → renamed STECON |
| `GULF` | `GULFI` | GULFI delisted Apr 2025 → renamed GULF |
| `CPAXT` | `CPAXTT` | CPAXTT delisted Oct 2024 → renamed CPAXT |
| `BSRC` | `SPRC` | SPRC renamed BSRC (Bangchak SRC) |
| `BKIH` | `BKI` | BKI delisted Feb 2024 → BKIH holding |

**Result**: ตาราง `setsmart_symbol_meta` (1 row/symbol):
- `symbol` — SETSMART symbol
- `eikon_alias` — symbol ที่ใช้หา Eikon (= symbol หรือ rename ตามตาราง)
- `market` (SET / mai), `delisted`, `delisted_code`, `is_trust`, `namechange`
- `eikon_ticker`, `latest_company_name`
- `in_eikon` (True ถ้า match Eikon ได้)

In [37]:
# Manual rename mapping (SETSMART symbol -> Eikon symbol)
# Add new entries here as user identifies them.
SETSMART_TO_EIKON_ALIAS = {
    "SCB":    "SCBB",
    "TRUE":   "TRUEE",
    "STECON": "STEC",
    "GULF":   "GULFI",
    "CPAXT":  "CPAXTT",
    "BSRC":   "SPRC",
    "BKIH":   "BKI",
    "F&D":    "FND",     # SETSMART uses F&D, Eikon uses FND
    "L&E":    "LNE",     # SETSMART uses L&E, Eikon uses LNE
    "S&J":    "SNJ",     # SETSMART uses S&J, Eikon uses SNJ
}

# Collect all unique SETSMART symbols from meeting + board panels
setsmart_symbols = pd.Series(
    pd.concat([setsmart_meeting["symbol"], setsmart_board["symbol"]]).dropna().unique(),
    name="symbol",
).to_frame()
print(f"Unique SETSMART symbols (from meeting + board): {len(setsmart_symbols)}")

# Apply manual rename: build eikon_alias column
setsmart_symbols["eikon_alias"] = setsmart_symbols["symbol"].map(
    lambda s: SETSMART_TO_EIKON_ALIAS.get(s, s)
)

# Left-join with Eikon symbol lookup on eikon_alias
setsmart_symbol_meta = setsmart_symbols.merge(
    eikon_symbol_lookup[["symbol", "market", "delisted", "delisted_code", "is_trust", "namechange",
                         "eikon_ticker", "latest_company_name"]].rename(columns={"symbol": "eikon_alias"}),
    on="eikon_alias",
    how="left",
)
setsmart_symbol_meta["in_eikon"] = setsmart_symbol_meta["eikon_ticker"].notna()

print(f"setsmart_symbol_meta shape: {setsmart_symbol_meta.shape}")
print()
print("Match coverage:")
print(f"  matched to Eikon : {setsmart_symbol_meta['in_eikon'].sum()}")
print(f"  unmatched        : {(~setsmart_symbol_meta['in_eikon']).sum()}")
print()
print("Market breakdown (matched only):")
print(setsmart_symbol_meta[setsmart_symbol_meta['in_eikon']].groupby(['market', 'delisted']).size().to_string())
print()
print("Manual renames applied (rows where symbol != eikon_alias):")
print(setsmart_symbol_meta[setsmart_symbol_meta['symbol'] != setsmart_symbol_meta['eikon_alias']][['symbol','eikon_alias','market','delisted','in_eikon']].to_string(index=False))
print()
unmatched = setsmart_symbol_meta[~setsmart_symbol_meta["in_eikon"]]["symbol"].tolist()
print(f"Still unmatched ({len(unmatched)}, likely new IPOs / not in Eikon dataset):")
print(f"  {unmatched}")

Unique SETSMART symbols (from meeting + board): 907
setsmart_symbol_meta shape: (907, 10)

Match coverage:
  matched to Eikon : 821
  unmatched        : 86

Market breakdown (matched only):
market  delisted
SET     False       612
        True         17
mai     False       190
        True          2

Manual renames applied (rows where symbol != eikon_alias):
symbol eikon_alias market delisted  in_eikon
   S&J         SNJ    SET    False      True
   SCB        SCBB    SET     True      True
  TRUE       TRUEE    SET     True      True
  BKIH         BKI    SET     True      True
 CPAXT      CPAXTT    SET     True      True
   F&D         FND    SET    False      True
  GULF       GULFI    SET     True      True
   L&E         LNE    SET    False      True
STECON        STEC    SET     True      True
  BSRC        SPRC    SET    False      True

Still unmatched (86, likely new IPOs / not in Eikon dataset):
  ['ACAP', 'ADVICE', 'ANI', 'BKGI', 'BLISS', 'BPS', 'CREDIT', 'EURO', 'GABLE', 

## C4. Build SETSMART master panel

Outer-merge `setsmart_meeting` + `setsmart_board` บน `(symbol, year)` แล้ว m:1 merge `setsmart_symbol_meta` บน `symbol` (เพื่อใส่ market/delisted จาก Eikon)

In [38]:
# Align dtypes
setsmart_meeting["year"] = setsmart_meeting["year"].astype("Int64")
setsmart_board["year"]   = setsmart_board["year"].astype("Int64")

# (a) Outer-merge meeting + board on (symbol, year)
master_setsmart = setsmart_meeting.merge(
    setsmart_board, on=["symbol", "year"], how="outer"
)

# (b) m:1 merge metadata (eikon_alias / market / delisted / eikon_ticker / in_eikon)
master_setsmart = master_setsmart.merge(setsmart_symbol_meta, on="symbol", how="left")

# Reorder: meta cols up front
front = ["symbol", "eikon_alias", "year", "market", "delisted", "delisted_code",
         "is_trust", "namechange", "in_eikon", "eikon_ticker", "latest_company_name"]
other = [c for c in master_setsmart.columns if c not in front]
master_setsmart = master_setsmart[front + other]

print(f"master_setsmart shape: {master_setsmart.shape}")
print(f"unique symbols: {master_setsmart['symbol'].nunique()}")
print()
print("By year:")
print(master_setsmart.groupby("year").size().to_string())
print()
print("Coverage per year (firm-years with non-null values):")
data_cols = ["num_meeting_setsmart", "avg_attent_meeting_pct_setsmart",
             "total_directors_setsmart", "independent_directors_setsmart", "pct_independent_setsmart"]
for col in data_cols:
    by_yr = master_setsmart.groupby("year")[col].apply(lambda x: x.notna().sum())
    print(f"  {col}: {dict(by_yr)}")
print()
print("Market breakdown of firm-years (from Eikon lookup):")
print(master_setsmart.groupby(["market", "delisted"], dropna=False).size().to_string())
print()
print("Sample ADVANC:")
print(master_setsmart[master_setsmart["symbol"]=="ADVANC"].sort_values("year")[
    ["symbol","eikon_alias","year","market","delisted","in_eikon",
     "num_meeting_setsmart","total_directors_setsmart","pct_independent_setsmart"]
].to_string())
print()
print("Sample rename: SCB (uses eikon_alias=SCBB):")
print(master_setsmart[master_setsmart["symbol"]=="SCB"].sort_values("year")[
    ["symbol","eikon_alias","year","market","delisted","in_eikon",
     "num_meeting_setsmart","total_directors_setsmart"]
].to_string())

master_setsmart shape: (4473, 23)
unique symbols: 907

By year:
year
2021    908
2022    908
2023    897
2024    884
2025    876

Coverage per year (firm-years with non-null values):
  num_meeting_setsmart: {np.int64(2021): np.int64(4), np.int64(2022): np.int64(659), np.int64(2023): np.int64(740), np.int64(2024): np.int64(803), np.int64(2025): np.int64(694)}
  avg_attent_meeting_pct_setsmart: {np.int64(2021): np.int64(4), np.int64(2022): np.int64(659), np.int64(2023): np.int64(740), np.int64(2024): np.int64(803), np.int64(2025): np.int64(694)}
  total_directors_setsmart: {np.int64(2021): np.int64(6), np.int64(2022): np.int64(812), np.int64(2023): np.int64(845), np.int64(2024): np.int64(862), np.int64(2025): np.int64(776)}
  independent_directors_setsmart: {np.int64(2021): np.int64(6), np.int64(2022): np.int64(812), np.int64(2023): np.int64(845), np.int64(2024): np.int64(862), np.int64(2025): np.int64(776)}
  pct_independent_setsmart: {np.int64(2021): np.int64(6), np.int64(2022): np.int

## C5. ตัด Trust/REIT/Fund ออกจาก `master_setsmart` ด้วย

ใช้ `is_trust` flag จาก Eikon lookup (ใน C3) — รวมถึงตัวที่ `is_trust=True` แต่ `in_eikon=True`

In [39]:
# is_trust comes back as object dtype after merge (True/False/NaN mix),
# which breaks ~. Cast to bool first.
trust_mask = master_setsmart['is_trust'].fillna(False).astype(bool)

n_trust_rows = trust_mask.sum()
n_trust_syms = master_setsmart.loc[trust_mask, 'symbol'].nunique()
print(f"Dropping {n_trust_syms} trust symbols ({n_trust_rows} firm-year rows from master_setsmart)")

master_setsmart = master_setsmart[~trust_mask].reset_index(drop=True)

print(f"After drop: master_setsmart {master_setsmart.shape}")
print(f"  unique symbols: {master_setsmart['symbol'].nunique()}")
print()
print("Market breakdown (firm-years after drop):")
print(master_setsmart.groupby(['market','delisted'], dropna=False).size().to_string())

Dropping 8 trust symbols (40 firm-year rows from master_setsmart)
After drop: master_setsmart (4433, 23)
  unique symbols: 899

Market breakdown (firm-years after drop):
market  delisted
SET     False       3023
        True          67
mai     False        950
        True           4
NaN     NaN          389


# Phase D — 1. SETSMART VS EIKON (compare + merge)

# Phase E — Save (CSV + .dta)

_will produce `master_panel_eikon.{csv,dta}`, `master_panel_setsmart.{csv,dta}`, `master_panel_combined.{csv,dta}`_